# SoundStream Inference Notebook

This notebook provides an example inference pipeline for an implementation of the SoundStream neural codec hosted [here](https://github.com/yakuri354/soundstream).

The notebook downloads pretrained checkpoints from HuggingFace.

## Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/yakuri354/soundstream soundstream
%cd soundstream

In [ ]:
!pip install -r inference-requirements.txt

## Audio link
You can test the model on any recording by pasting a link to a publicly accessible audio file into the cell below.

In [13]:
LINK = "https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav"

## Loading the model

Download the model from HuggingFace. The intermediate training checkpoints are also stored on HF, they are accessible as different revisions of the same repo (i.e. epoch4)

In [ ]:
from src.model.soundstream import SoundStream
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

model = SoundStream.from_pretrained("yakuri354-2/soundstream").eval()

## Loading the audio

We load the file from the provided URL using torchaudio and play it through Jupyter

The model only supports 16kHz audio, so we need to resample it.

In [15]:
import torchaudio
from IPython.display import Audio

SAMPLE_RATE = 16000

audio, orig_rate = torchaudio.load(LINK)

audio = torchaudio.transforms.Resample(orig_rate, SAMPLE_RATE)(audio)

Audio(audio, rate=SAMPLE_RATE)

## Inference

We encode and decode the audio using the model. Note: inference happens entirely on CPU.

In [ ]:
import torch

with torch.no_grad():
    outputs = model(audio)

In [ ]:
dec_audio = outputs["decoded"].cpu().detach()

Audio(dec_audio, rate=SAMPLE_RATE)

## Compression

Here we compare the bitrates of the original and compressed audio.

In [ ]:
from math import log2

raw_bitrate = audio.element_size() * audio.nelement() * 8 / SAMPLE_RATE
enc_bitrate = (
    outputs["encoded"].nelement() * log2(model.rvq.k) * model.rvq.d / SAMPLE_RATE
)

print("Original bitrate (raw):", raw_bitrate, "bits per second")
print("Compressed bitrate:", enc_bitrate, "bits per second")
print(f"Compressed is {enc_bitrate / raw_bitrate:.1f} of raw")

Original bitrate (raw): 268.694 bits per second
Compressed bitrate: 26.84 bits per second
Compressed is 0.1 of raw


As we can see, the model reduces the bitrate by a significant margin